# 04 The baseline: a simple model on the same 14 numbers

**What a baseline is.** Before I trust the forest, I need something to compare it with. A score on its own says
nothing: is 85% recall good? Only if something simpler scores clearly lower. So the baseline is the simplest real
model I can train on exactly the same input. If the forest can't clearly beat it, 300 trees are not worth it.

**Why not "always the most common label".** That was the first plan. MixedWM38 was balanced on purpose, so
almost every type has exactly the same number of training wafers, and there is no most common label. The tie goes
to Normal, which never says yes, so it scores 0% recall on every defect. Any model that says yes once beats that,
so as a bar it proves nothing.

**What this notebook does.** It trains the baseline, logistic regression on the 14 features from notebook 03, and
scores it on the validation part. Next to it are two guesses that never look at a wafer: they show what luck scores,
and why the comparison uses F1 and not recall alone.

**In:** `output/features.npz` (from notebook 03) **Out:** `output/simple_model.joblib`, `output/baseline_validation.csv`

## 1. Load the features

**Why this step:** The baseline gets exactly what the forest gets, so the comparison is fair.

**The problem:** Load the training and validation features and their answers.

**How:** Notebook 03 saved everything in `output/features.npz`. `X` is the usual name for the input, 14 numbers per wafer. `y` is the answers: 8 positions per wafer, one per defect, in the order Center, Donut, Edge-Loc, Edge-Ring, Loc, Near-Full, Scratch, Random. A 1 means that defect is there.\
`allow_pickle=True` is needed because the file also holds the feature names as text. The test part stays closed until notebook 06.

In [1]:
import csv
import os

import numpy as np
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier

import wafer          # my own file with the shared functions: label_name, the scores

f = np.load("output/features.npz", allow_pickle=True)
X_train = f["X_train"]
y_train = f["y_train"]
X_val = f["X_val"]
y_val = f["y_val"]

print("Training:", X_train.shape, "answers", y_train.shape)
print("Validation:", X_val.shape, "answers", y_val.shape)

Training: (25822, 14) answers (25822, 8)
Validation: (5533, 14) answers (5533, 8)


**What you should see:** The same sizes as notebook 03 printed: 14 numbers and 8 answers per wafer.

## 2. How to score

**Why this step:** The baseline, the guesses and the forest all get the same scores, so they can stand side by side.

**The problem:** Turn a table of answers into scores per defect.

**How:** **Recall** answers: of the wafers that really have a defect, how many did it catch? 10 wafers really have a Scratch and it says Scratch on 8 of them: recall is 8 / 10 = 80%. A missed defect in a fab means broken chips go on.\
**Precision** answers the opposite: of the wafers it said have a Scratch, how many really did? It said Scratch 10 times and 8 were right: precision is 80%. The wrong ones are false alarms, an engineer looks for nothing.\
**F1** puts the two in one number: `2 × precision × recall / (precision + recall)`. It's only high when *both* are high. 100% recall with 10% precision gives an F1 of 18%, not 55% like the plain average would. Step 4 shows why I need it.\
**All 8 right** is strict: the share of wafers where all 8 positions are correct.\
`score(name, y_pred)` is a small function of mine: it scores one set of answers on the validation part, prints the scores and remembers them for the file in step 7. The counting itself is in `wafer.py`, written with plain loops so every step can be checked. When something never says yes to a defect, precision would be 0 / 0, and `wafer.py` makes that 0.

In [2]:
scored = []    # name, recall, precision, F1, all 8 right. Saved in step 7


def score(name, y_pred):
    recall, precision = wafer.scores_per_defect(y_val, y_pred)
    f1 = wafer.f1_per_defect(recall, precision)
    all_right = wafer.share_all_eight_right(y_val, y_pred)
    scored.append([name, recall, precision, f1, all_right])

    print(name)
    for i in range(len(wafer.DEFECTS)):
        print(f"  {wafer.DEFECTS[i]}: recall {recall[i] * 100:.1f}%, precision {precision[i] * 100:.1f}%, "
              f"F1 {f1[i] * 100:.1f}%")
    print(f"  All 8 right: {all_right * 100:.1f}%")

## 3. The floor: guess by share

**Why this step:** Before the baseline, I want to know what luck alone scores. A model close to this learned almost nothing.

**The problem:** Per defect, say yes on a random part of the wafers as big as that defect's share in training, without looking at the wafer.

**How:** `y_train.mean(axis=0)` takes the average of every column. A column is only 0s and 1s, so the average is the share of wafers with that defect. 0.33 means one in three.\
`rng.random(y_val.shape)` gives a random number between 0 and 1 for every position of every validation wafer. `< shares` compares every column with its own share, so a defect on 33% of the wafers gets a yes about 33% of the time. Seed 42 again, so the guess is the same every run (QC-06).

In [3]:
# Per defect, the share of training wafers that have it
shares = y_train.mean(axis=0)
for i in range(len(wafer.DEFECTS)):
    print(f"{wafer.DEFECTS[i]}: on {shares[i] * 100:.1f}% of the training wafers")
print()

rng = np.random.default_rng(42)
by_share = (rng.random(y_val.shape) < shares).astype(y_val.dtype)
score("Guess by share", by_share)

Center: on 32.5% of the training wafers
Donut: on 32.5% of the training wafers
Edge-Loc: on 32.5% of the training wafers
Edge-Ring: on 32.5% of the training wafers
Loc: on 48.8% of the training wafers
Near-Full: on 0.4% of the training wafers
Scratch: on 48.8% of the training wafers
Random: on 2.0% of the training wafers

Guess by share
  Center: recall 32.8%, precision 32.5%, F1 32.6%
  Donut: recall 31.7%, precision 32.2%, F1 32.0%
  Edge-Loc: recall 31.8%, precision 31.8%, F1 31.8%
  Edge-Ring: recall 32.5%, precision 31.7%, F1 32.1%
  Loc: recall 47.7%, precision 48.4%, F1 48.0%
  Near-Full: recall 0.0%, precision 0.0%, F1 0.0%
  Scratch: recall 48.2%, precision 49.3%, F1 48.7%
  Random: recall 1.8%, precision 1.7%, F1 1.7%
  All 8 right: 1.9%


**What you should see:** The share per defect, then the scores. A random guess at the right rate gets recall,
precision and F1 of about that same share: it's right as often as the defect is there, by luck. That's the floor.

## 4. Why F1: always yes

**Why this step:** If saying yes to everything scores well on some number, that number can be bought without learning anything.

**The problem:** Say every defect is on every wafer, and score it.

**How:** `np.ones_like(y_val)` makes a table the same shape as `y_val`, full of 1s.

In [4]:
always_yes = np.ones_like(y_val)
score("Always yes", always_yes)

Always yes
  Center: recall 100.0%, precision 32.5%, F1 49.1%
  Donut: recall 100.0%, precision 32.5%, F1 49.1%
  Edge-Loc: recall 100.0%, precision 32.5%, F1 49.1%
  Edge-Ring: recall 100.0%, precision 32.5%, F1 49.1%
  Loc: recall 100.0%, precision 48.8%, F1 65.6%
  Near-Full: recall 100.0%, precision 0.4%, F1 0.7%
  Scratch: recall 100.0%, precision 48.8%, F1 65.6%
  Random: recall 100.0%, precision 2.0%, F1 4.0%
  All 8 right: 0.0%


**What you should see:** 100% recall on every defect, because nothing is ever missed. Precision is the share of
wafers that really have that defect, and all 8 right is 0. So recall on its own can be bought by saying yes to
everything. That's why QC-04 compares F1: it stays low here.

## 5. The baseline: logistic regression

**Why this step:** This is the model the forest has to beat (QC-04). It looks at the wafer, through the same 14 numbers, but in the simplest way there is.

**The problem:** Train it on the training part and score it on validation.

**How:** **Logistic regression.** Per defect it gives every one of the 14 numbers a weight, adds them up, and says yes when the sum is above a cutoff. That draws one straight line through the data, where a forest can draw many small boxes. So a defect the baseline gets right is one the 14 numbers make easy. Where the forest pulls away later, it needs those boxes.\
`LogisticRegression(max_iter=1000)`: `max_iter` is how many steps it may take to find the weights, the default of 100 can stop too early. It answers one yes/no question, so `MultiOutputClassifier` trains one for every defect, 8 in all. The 14 numbers are all shares between 0 and 1 already, so they don't need scaling first.\
`simple.fit(X_train, y_train)` is the training, `simple.predict(X_val)` gives its answers, 8 columns of 0 or 1.

In [5]:
simple = MultiOutputClassifier(LogisticRegression(max_iter=1000))
simple.fit(X_train, y_train)

score("Simple model (baseline)", simple.predict(X_val))

Simple model (baseline)
  Center: recall 82.9%, precision 90.4%, F1 86.5%
  Donut: recall 83.2%, precision 92.6%, F1 87.7%
  Edge-Loc: recall 0.0%, precision 0.0%, F1 0.0%
  Edge-Ring: recall 94.7%, precision 89.2%, F1 91.8%
  Loc: recall 78.1%, precision 87.6%, F1 82.6%
  Near-Full: recall 95.0%, precision 100.0%, F1 97.4%
  Scratch: recall 56.2%, precision 66.2%, F1 60.8%
  Random: recall 4.4%, precision 20.0%, F1 7.2%
  All 8 right: 31.1%


**What you should see:** The scores of the baseline. Look for the defects where it scores near zero: a straight line
can't separate those with these 14 numbers. Look also where it scores high: those are the defects where the forest
will have a hard time pulling away.

## 6. Side by side

**Why this step:** One table shows where the baseline stands between luck and a good score.

**The problem:** Print F1 per defect, and all 8 right, for all three.

**How:** One line per defect, one column per entry in `scored`. `:>16` in an f-string pads text to 16 characters, so the columns line up.

In [6]:
header = f"{'F1':<11}"
for name, recall, precision, f1, all_right in scored:
    header += f"{name:>26}"
print(header)

for i in range(len(wafer.DEFECTS)):
    line = f"{wafer.DEFECTS[i]:<11}"
    for name, recall, precision, f1, all_right in scored:
        line += f"{f1[i] * 100:25.1f}%"
    print(line)

line = f"{'All 8 right':<11}"
for name, recall, precision, f1, all_right in scored:
    line += f"{all_right * 100:25.1f}%"
print(line)

F1                     Guess by share                Always yes   Simple model (baseline)
Center                          32.6%                     49.1%                     86.5%
Donut                           32.0%                     49.1%                     87.7%
Edge-Loc                        31.8%                     49.1%                      0.0%
Edge-Ring                       32.1%                     49.1%                     91.8%
Loc                             48.0%                     65.6%                     82.6%
Near-Full                        0.0%                      0.7%                     97.4%
Scratch                         48.7%                     65.6%                     60.8%
Random                           1.7%                      4.0%                      7.2%
All 8 right                      1.9%                      0.0%                     31.1%


**What you should see:** The baseline above the guess by share on every defect where the 14 numbers carry
information, and close to it where they don't. Always yes has the highest recall of all three and still a low F1.

## 7. Save

**Why this step:** Notebook 06 scores the baseline on the test part, and notebook 08 checks against saved files. Copying numbers off the screen by hand is how mistakes get in.

**The problem:** Save the trained baseline, and every score in a file.

**How:** `joblib.dump` saves the trained model. It's 8 rows of 14 weights, so the file is tiny.\
The scores go in one row per name, defect and score, so the file has the same columns whatever is in it: `baseline`, `defect`, `score`, `value`. All 8 right goes in as defect `all`. `csv.writer` writes one list per row, `round(..., 4)` keeps 4 decimals.

In [7]:
os.makedirs("output", exist_ok=True)
joblib.dump(simple, "output/simple_model.joblib")
print("Saved output/simple_model.joblib")

os.makedirs("output", exist_ok=True)
with open("output/baseline_validation.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["baseline", "defect", "score", "value"])
    for name, recall, precision, f1, all_right in scored:
        for i in range(len(wafer.DEFECTS)):
            writer.writerow([name, wafer.DEFECTS[i], "recall", round(recall[i], 4)])
            writer.writerow([name, wafer.DEFECTS[i], "precision", round(precision[i], 4)])
            writer.writerow([name, wafer.DEFECTS[i], "f1", round(f1[i], 4)])
        writer.writerow([name, "all", "all 8 right", round(all_right, 4)])
print("Saved output/baseline_validation.csv")

Saved output/simple_model.joblib
Saved output/baseline_validation.csv


**What you should see:** Two "Saved" lines.

## What this means for the next notebooks

The baseline is trained and saved. Notebook 05 trains the forest on the same 14 numbers and puts its validation
score next to the baseline's. Notebook 06 scores the forest, the baseline and the guess by share on the test part,
and QC-04 asks that the forest has a higher F1 than the baseline on every one of the 8 defects.

## Functions in this notebook

| Function | What it does | Where to read more |
|---|---|---|
| `np.load` | opens a saved .npz file | [docs](https://numpy.org/doc/stable/reference/generated/numpy.load.html) |
| `.mean(axis=0)` | the average of every column | [docs](https://numpy.org/doc/stable/reference/generated/numpy.mean.html) |
| `rng.random` | random numbers between 0 and 1 | [docs](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.random.html) |
| `np.ones_like` | a table of the same shape, full of 1s | [docs](https://numpy.org/doc/stable/reference/generated/numpy.ones_like.html) |
| `LogisticRegression` | the baseline: a weighted sum of the 14 numbers and a cutoff | [docs](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) |
| `MultiOutputClassifier` | trains one yes/no model per defect | [docs](https://scikit-learn.org/stable/modules/generated/sklearn.multioutput.MultiOutputClassifier.html) |
| `joblib.dump` | saves the trained model to a file | [docs](https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html) |
| `csv.writer` | writes rows to a .csv file | [docs](https://docs.python.org/3/library/csv.html) |

Why a baseline should be the strongest simple thing and not the weakest, and the no-look guesses scikit-learn
has built in: [Dummy estimators](https://scikit-learn.org/stable/modules/model_evaluation.html#dummy-estimators).